In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import joblib

!pip install torch_geometric

from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.0 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DATN/network_dataset_v2_7500.csv")

df_normal_all = df[df['class'] == 'normal'].reset_index(drop=True)
df_attacks = df[df['class'] != 'normal'].reset_index(drop=True)
attack_types = df_attacks['class'].unique()

df_normal_all = df_normal_all.sample(frac=1, random_state=42).reset_index(drop=True)

df_normal_train = df_normal_all.iloc[:500].reset_index(drop=True)
df_normal_test  = df_normal_all.iloc[500:600].reset_index(drop=True)

print(f"Train normal: {len(df_normal_train)}")
print(f"Test normal: {len(df_normal_test)}")

Train normal: 500
Test normal: 100


In [ ]:
groups = {
    'Interface': ['ifInOctets11','ifOutOctets11','ifoutDiscards11','ifInUcastPkts11',
                  'ifInNUcastPkts11','ifInDiscards11','ifOutUcastPkts11','ifOutNUcastPkts11'],
    'IP': ['ipInReceives','ipInDelivers','ipOutRequests','ipOutDiscards',
           'ipInDiscards','ipForwDatagrams','ipOutNoRoutes','ipInAddrErrors'],
    'TCP': ['tcpOutRsts','tcpInSegs','tcpOutSegs','tcpPassiveOpens',
            'tcpRetransSegs','tcpCurrEstab','tcpEstabResets','tcp?ActiveOpens'],
    'UDP': ['udpInDatagrams','udpOutDatagrams','udpInErrors','udpNoPorts'],
    'ICMP': ['icmpInMsgs','icmpInDestUnreachs','icmpOutMsgs',
             'icmpOutDestUnreachs','icmpInEchos','icmpOutEchoReps']
}

if 'tcp?ActiveOpens' in df.columns:
    df['tcpActiveOpens'] = df['tcp?ActiveOpens']

group_cols = {
    'Interface': groups['Interface'],
    'IP': groups['IP'],
    'TCP': groups['TCP'] + (['tcpActiveOpens'] if 'tcpActiveOpens' in df.columns else []),
    'UDP': groups['UDP'],
    'ICMP': groups['ICMP']
}

node_order = ['Interface','IP','TCP','UDP','ICMP']
max_dim = max(len(cols) for cols in group_cols.values())

In [ ]:
def create_features(df_set, N, max_dim, group_cols, node_order, scalers=None, fit=False):
    x_padded = torch.zeros(N, 5, max_dim)

    if fit:
        scalers = {}

    for i, group in enumerate(node_order):
        cols = group_cols[group]
        existing_cols = [c for c in cols if c in df_set.columns]
        group_df = df_set[existing_cols].values.astype(float)

        if fit:
            scaler = StandardScaler()
            scaled = scaler.fit_transform(group_df)
            scalers[group] = scaler
        else:
            scaler = scalers[group]
            scaled = scaler.transform(group_df)

        feats = torch.tensor(scaled[:N], dtype=torch.float)
        dim = feats.shape[1]
        x_padded[:, i, :dim] = feats

    return x_padded, scalers


N_normal = len(df_normal_train)

x_normal, scalers_normal = create_features(
    df_normal_train, N_normal, max_dim, group_cols, node_order, fit=True
)

time_emb = torch.zeros(N_normal, 5, 1)
for t in range(N_normal):
    time_emb[t] = torch.sin(torch.tensor(t / N_normal * 2 * np.pi).float())

x_normal = torch.cat([x_normal, time_emb], dim=-1)

in_channels = x_normal.shape[-1]

In [ ]:
edge_index = torch.tensor([
    [0,1],[1,0],
    [1,2],[2,1],
    [1,3],[3,1],
    [1,4],[4,1]
]).t().long()

adj_target = torch.zeros(5,5)
adj_target[edge_index[0], edge_index[1]] = 1

data_list_normal = [
    Data(x=x_normal[t], edge_index=edge_index.clone())
    for t in range(N_normal)
]

In [ ]:
class GCN_AutoEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        emb = self.conv3(x, edge_index)

        adj_logits = torch.matmul(emb, emb.t())

        return emb, adj_logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCN_AutoEncoder(
    in_channels=in_channels,
    hidden_channels=256,
    out_channels=128
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

criterion = nn.BCEWithLogitsLoss()

loader = DataLoader(data_list_normal, batch_size=256, shuffle=True)

best_loss = float('inf')

print("Training GCN on normal data...")

for epoch in range(200):

    total_loss = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        emb, adj_logits = model(batch)

        batch_size = batch.num_graphs
        adj_target_batch = torch.block_diag(
            *([adj_target.to(device)] * batch_size)
        )

        loss = criterion(adj_logits, adj_target_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    scheduler.step(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/DATN/best_gcn_model.pth")

print("Training done.")

Training GCN on normal data...
Epoch 0 | Loss: 0.7994
Epoch 10 | Loss: 0.6945
Epoch 20 | Loss: 0.6933
Epoch 30 | Loss: 0.6932
Epoch 40 | Loss: 0.6932
Epoch 50 | Loss: 0.6931
Epoch 60 | Loss: 0.6931
Epoch 70 | Loss: 0.6932
Epoch 80 | Loss: 0.6931
Epoch 90 | Loss: 0.6931
Epoch 100 | Loss: 0.6931
Epoch 110 | Loss: 0.6932
Epoch 120 | Loss: 0.6931
Epoch 130 | Loss: 0.6931
Epoch 140 | Loss: 0.6931
Epoch 150 | Loss: 0.6931
Epoch 160 | Loss: 0.6931
Epoch 170 | Loss: 0.6931
Epoch 180 | Loss: 0.6931
Epoch 190 | Loss: 0.6931
Training done.


In [ ]:
model.eval()

normal_scores = []

with torch.no_grad():
    for data in data_list_normal:
        data = data.to(device)
        _, adj_logits = model(data)

        recon_score = F.binary_cross_entropy_with_logits(
            adj_logits, adj_target.to(device)
        ).item()

        normal_scores.append(recon_score)

mean = np.mean(normal_scores)
std = np.std(normal_scores)
threshold = mean + 2 * std

print("Threshold:", threshold)

Threshold: 0.6995548165639247


In [ ]:
# ===== TÍNH RECALL CHO TỪNG ATTACK =====

for attack_type in attack_types:

    df_type = df_attacks[df_attacks['class'] == attack_type].reset_index(drop=True)
    N_type = len(df_type)

    x_type, _ = create_features(
        df_type, N_type, max_dim, group_cols, node_order,
        scalers=scalers_normal, fit=False
    )

    time_emb_type = torch.zeros(N_type, 5, 1)
    for t in range(N_type):
        time_emb_type[t] = torch.sin(torch.tensor(t / N_type * 2 * np.pi).float())

    x_type = torch.cat([x_type, time_emb_type], dim=-1)

    data_list_type = [
        Data(x=x_type[t], edge_index=edge_index.clone())
        for t in range(N_type)
    ]

    detected = 0
    anomaly_scores = []

    with torch.no_grad():
        for t in range(N_type):

            data_test_type = data_list_type[t].to(device)
            _, adj_logits = model(data_test_type)

            recon_score_type = F.binary_cross_entropy_with_logits(
                adj_logits, adj_target.to(device)
            ).item()

            anomaly_scores.append(recon_score_type)

            if recon_score_type > threshold:
                detected += 1

    recall = detected / N_type

    print(f"\n===== {attack_type} =====")
    print(f"Tổng mẫu: {N_type}")
    print(f"Phát hiện: {detected}")
    print(f"Recall: {recall:.4f}")
    print(f"Mean anomaly score: {np.mean(anomaly_scores):.4f}")


===== icmp-echo =====
Tổng mẫu: 632
Phát hiện: 619
Recall: 0.9794
Mean anomaly score: 6342604.0411

===== tcp-syn =====
Tổng mẫu: 960
Phát hiện: 960
Recall: 1.0000
Mean anomaly score: 10597804.5659

===== udp-flood =====
Tổng mẫu: 773
Phát hiện: 773
Recall: 1.0000
Mean anomaly score: 13051884.8314

===== httpFlood =====
Tổng mẫu: 573
Phát hiện: 573
Recall: 1.0000
Mean anomaly score: 1720890.4505

===== slowloris =====
Tổng mẫu: 780
Phát hiện: 675
Recall: 0.8654
Mean anomaly score: 1.0909

===== slowpost =====
Tổng mẫu: 480
Phát hiện: 480
Recall: 1.0000
Mean anomaly score: 21157618.6762

===== bruteForce =====
Tổng mẫu: 200
Phát hiện: 200
Recall: 1.0000
Mean anomaly score: 14901238.3991


In [ ]:
# ===== TEST NORMAL SET =====

df_type = df_normal_test.reset_index(drop=True)
N_type = len(df_type)

x_type, _ = create_features(
    df_type, N_type, max_dim, group_cols, node_order,
    scalers=scalers_normal, fit=False
)

time_emb_type = torch.zeros(N_type, 5, 1)
for t in range(N_type):
    time_emb_type[t] = torch.sin(torch.tensor(t / N_type * 2 * np.pi).float())

x_type = torch.cat([x_type, time_emb_type], dim=-1)

data_list_type = [
    Data(x=x_type[t], edge_index=edge_index.clone())
    for t in range(N_type)
]

detected = 0
anomaly_scores = []

with torch.no_grad():
    for t in range(N_type):

        data_test_type = data_list_type[t].to(device)
        _, adj_logits = model(data_test_type)

        recon_score_type = F.binary_cross_entropy_with_logits(
            adj_logits, adj_target.to(device)
        ).item()

        anomaly_scores.append(recon_score_type)

        if recon_score_type > threshold:
            detected += 1

fpr = detected / N_type

print(f"\n===== NORMAL TEST SET =====")
print(f"Tổng mẫu: {N_type}")
print(f"Bị phát hiện nhầm (False Positive): {detected}")
print(f"False Positive Rate (FPR): {fpr:.4f}")
print(f"Mean anomaly score: {np.mean(anomaly_scores):.4f}")


===== NORMAL TEST SET =====
Tổng mẫu: 100
Bị phát hiện nhầm (False Positive): 3
False Positive Rate (FPR): 0.0300
Mean anomaly score: 0.6958
